## Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import joblib
try:
    import seaborn as sns
except:
    ! pip install seaborn
    import seaborn as sns
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
except:
    ! pip install torch
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
try:
    import optuna
except:
    ! pip install optuna
    import optuna
try:
    import pyarrow
except:
    ! pip install pyarrow
    import pyarrow

optuna.logging.set_verbosity(optuna.logging.WARNING)

## Helper Classes and Functions

In [ ]:
class RatingsDataset(Dataset):
    """PyTorch Dataset for user-item ratings."""

    def __init__(self, arr_users, arr_items, arr_ratings):
        self.arr_users = torch.LongTensor(arr_users)
        self.arr_items = torch.LongTensor(arr_items)
        self.arr_ratings = torch.FloatTensor(arr_ratings)

    def __len__(self):
        return len(self.arr_ratings)

    def __getitem__(self, idx):
        return self.arr_users[idx], self.arr_items[idx], self.arr_ratings[idx]

In [ ]:
class NCFModel(nn.Module):
    """Neural Collaborative Filtering model combining GMF and MLP paths."""

    def __init__(self, int_n_users, int_n_items, int_embed_dim, list_hidden_dims, flt_dropout=0.2):
        super(NCFModel, self).__init__()

        # GMF path embeddings
        self.gmf_user_embedding = nn.Embedding(int_n_users, int_embed_dim)
        self.gmf_item_embedding = nn.Embedding(int_n_items, int_embed_dim)

        # MLP path embeddings
        self.mlp_user_embedding = nn.Embedding(int_n_users, int_embed_dim)
        self.mlp_item_embedding = nn.Embedding(int_n_items, int_embed_dim)

        # MLP layers
        list_layers = []
        int_input_dim = int_embed_dim * 2
        for int_hidden_dim in list_hidden_dims:
            list_layers.append(nn.Linear(int_input_dim, int_hidden_dim))
            list_layers.append(nn.ReLU())
            list_layers.append(nn.Dropout(flt_dropout))
            int_input_dim = int_hidden_dim
        self.mlp_layers = nn.Sequential(*list_layers)

        # final prediction layer: GMF output (embed_dim) + MLP output (last hidden dim)
        int_final_dim = int_embed_dim + list_hidden_dims[-1]
        self.output_layer = nn.Linear(int_final_dim, 1)

        # initialize weights
        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.gmf_user_embedding.weight, std=0.01)
        nn.init.normal_(self.gmf_item_embedding.weight, std=0.01)
        nn.init.normal_(self.mlp_user_embedding.weight, std=0.01)
        nn.init.normal_(self.mlp_item_embedding.weight, std=0.01)
        for layer in self.mlp_layers:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                nn.init.zeros_(layer.bias)
        nn.init.xavier_uniform_(self.output_layer.weight)
        nn.init.zeros_(self.output_layer.bias)

    def forward(self, tensor_users, tensor_items):
        # GMF path: element-wise product
        gmf_user = self.gmf_user_embedding(tensor_users)
        gmf_item = self.gmf_item_embedding(tensor_items)
        gmf_out = gmf_user * gmf_item

        # MLP path: concatenation -> MLP
        mlp_user = self.mlp_user_embedding(tensor_users)
        mlp_item = self.mlp_item_embedding(tensor_items)
        mlp_input = torch.cat([mlp_user, mlp_item], dim=-1)
        mlp_out = self.mlp_layers(mlp_input)

        # combine GMF and MLP
        concat = torch.cat([gmf_out, mlp_out], dim=-1)
        output = self.output_layer(concat).squeeze(-1)
        return output

In [ ]:
class NCFRecommender:
    """Neural Collaborative Filtering recommender wrapper."""

    def __init__(self, int_n_users, int_n_items, int_embed_dim, list_hidden_dims,
                 flt_dropout=0.2, flt_lr=0.001, int_epochs=15, int_batch_size=512,
                 int_patience=3, str_device='cpu'):
        self.int_n_users = int_n_users
        self.int_n_items = int_n_items
        self.int_embed_dim = int_embed_dim
        self.list_hidden_dims = list_hidden_dims
        self.flt_dropout = flt_dropout
        self.flt_lr = flt_lr
        self.int_epochs = int_epochs
        self.int_batch_size = int_batch_size
        self.int_patience = int_patience
        self.str_device = str_device

        self.cls_model = NCFModel(
            int_n_users=int_n_users,
            int_n_items=int_n_items,
            int_embed_dim=int_embed_dim,
            list_hidden_dims=list_hidden_dims,
            flt_dropout=flt_dropout,
        ).to(str_device)

    def fit(self, df_train, df_valid, dict_user_to_idx, dict_item_to_idx):
        print(f'Training NCF (embed_dim={self.int_embed_dim}, hidden={self.list_hidden_dims}, '
              f'lr={self.flt_lr}, batch_size={self.int_batch_size})...')

        self.dict_user_to_idx = dict_user_to_idx
        self.dict_item_to_idx = dict_item_to_idx
        self.dict_idx_to_item = {v: k for k, v in dict_item_to_idx.items()}

        # create datasets
        ds_train = RatingsDataset(
            arr_users=df_train['user_idx'].values,
            arr_items=df_train['item_idx'].values,
            arr_ratings=df_train['rating'].values.astype(np.float32),
        )
        ds_valid = RatingsDataset(
            arr_users=df_valid['user_idx'].values,
            arr_items=df_valid['item_idx'].values,
            arr_ratings=df_valid['rating'].values.astype(np.float32),
        )

        bool_pin = (self.str_device != 'cpu')
        dl_train = DataLoader(ds_train, batch_size=self.int_batch_size, shuffle=True,
                              num_workers=2, pin_memory=bool_pin)
        dl_valid = DataLoader(ds_valid, batch_size=self.int_batch_size, shuffle=False,
                              num_workers=2, pin_memory=bool_pin)

        optimizer = torch.optim.Adam(self.cls_model.parameters(), lr=self.flt_lr)
        criterion = nn.MSELoss()

        self.list_train_loss = []
        self.list_valid_loss = []
        flt_best_valid_loss = np.inf
        int_no_improve = 0
        dict_best_state = None

        for int_epoch in tqdm(range(self.int_epochs), desc='Training'):
            # train
            self.cls_model.train()
            list_batch_loss = []
            for batch_users, batch_items, batch_ratings in dl_train:
                batch_users = batch_users.to(self.str_device)
                batch_items = batch_items.to(self.str_device)
                batch_ratings = batch_ratings.to(self.str_device)

                optimizer.zero_grad()
                predictions = self.cls_model(batch_users, batch_items)
                loss = criterion(predictions, batch_ratings)
                loss.backward()
                optimizer.step()
                list_batch_loss.append(loss.item())

            flt_train_loss = np.mean(list_batch_loss)
            self.list_train_loss.append(flt_train_loss)

            # validate
            self.cls_model.eval()
            list_batch_loss = []
            with torch.no_grad():
                for batch_users, batch_items, batch_ratings in dl_valid:
                    batch_users = batch_users.to(self.str_device)
                    batch_items = batch_items.to(self.str_device)
                    batch_ratings = batch_ratings.to(self.str_device)

                    predictions = self.cls_model(batch_users, batch_items)
                    loss = criterion(predictions, batch_ratings)
                    list_batch_loss.append(loss.item())

            flt_valid_loss = np.mean(list_batch_loss)
            self.list_valid_loss.append(flt_valid_loss)

            # early stopping
            if flt_valid_loss < flt_best_valid_loss:
                flt_best_valid_loss = flt_valid_loss
                int_no_improve = 0
                dict_best_state = self.cls_model.state_dict().copy()
            else:
                int_no_improve += 1
                if int_no_improve >= self.int_patience:
                    print(f'  Early stopping at epoch {int_epoch + 1}')
                    break

        # restore best model
        if dict_best_state is not None:
            self.cls_model.load_state_dict(dict_best_state)

        # store seen items per user
        self.dict_user_seen = df_train.groupby('user_id')['movie_id'].apply(set).to_dict()

        print(f'  Training complete. Best valid loss: {flt_best_valid_loss:.4f}')
        return self

    def predict(self, arr_user_ids, arr_movie_ids):
        self.cls_model.eval()

        # map IDs to indices, defaulting to 0 for unknown
        arr_user_idx = np.array([self.dict_user_to_idx.get(u, 0) for u in arr_user_ids])
        arr_item_idx = np.array([self.dict_item_to_idx.get(m, 0) for m in arr_movie_ids])

        tensor_users = torch.LongTensor(arr_user_idx).to(self.str_device)
        tensor_items = torch.LongTensor(arr_item_idx).to(self.str_device)

        with torch.no_grad():
            arr_predictions = self.cls_model(tensor_users, tensor_items).cpu().numpy()

        return np.clip(arr_predictions, 1.0, 5.0)

    def recommend_top_k(self, int_user_id, int_k_items=10):
        if int_user_id not in self.dict_user_to_idx:
            return []

        self.cls_model.eval()
        int_u_idx = self.dict_user_to_idx[int_user_id]
        set_seen = self.dict_user_seen.get(int_user_id, set())

        # score all items
        tensor_user = torch.LongTensor([int_u_idx] * self.int_n_items).to(self.str_device)
        tensor_items = torch.LongTensor(list(range(self.int_n_items))).to(self.str_device)

        with torch.no_grad():
            arr_scores = self.cls_model(tensor_user, tensor_items).cpu().numpy()

        # mask seen items
        for int_movie_id in set_seen:
            if int_movie_id in self.dict_item_to_idx:
                arr_scores[self.dict_item_to_idx[int_movie_id]] = -np.inf

        # get top-k
        arr_top_idx = np.argsort(-arr_scores)[:int_k_items]
        list_recs = [self.dict_idx_to_item[idx] for idx in arr_top_idx]
        return list_recs

In [ ]:
class RecommenderEvaluator:
    """Evaluator for recommender systems with rating and ranking metrics."""

    def __init__(self, int_k_at=10, flt_relevance_threshold=4.0, int_max_users=1000):
        self.int_k_at = int_k_at
        self.flt_relevance_threshold = flt_relevance_threshold
        self.int_max_users = int_max_users

    def compute_rating_metrics(self, arr_y_true, arr_y_pred):
        flt_rmse = np.sqrt(np.mean((arr_y_true - arr_y_pred) ** 2))
        flt_mae = np.mean(np.abs(arr_y_true - arr_y_pred))
        return {'flt_rmse': flt_rmse, 'flt_mae': flt_mae}

    def compute_ranking_metrics(self, df_test, cls_model):
        int_k = self.int_k_at
        flt_threshold = self.flt_relevance_threshold

        # get relevant items per user in test set
        df_relevant = df_test[df_test['rating'] >= flt_threshold]
        dict_user_relevant = df_relevant.groupby('user_id')['movie_id'].apply(set).to_dict()

        list_users = list(dict_user_relevant.keys())

        # sample users if too many to speed up evaluation
        if len(list_users) > self.int_max_users:
            list_users = list(np.random.RandomState(42).choice(
                list_users, size=self.int_max_users, replace=False,
            ))

        list_precision = []
        list_recall = []
        list_ndcg = []
        list_ap = []
        list_hit = []
        set_recommended_items = set()

        for int_user_id in tqdm(list_users, desc='Computing ranking metrics'):
            list_recs = cls_model.recommend_top_k(int_user_id, int_k_items=int_k)
            if len(list_recs) == 0:
                continue

            set_relevant = dict_user_relevant[int_user_id]
            set_recommended_items.update(list_recs)

            # precision@k
            int_hits = len(set(list_recs) & set_relevant)
            flt_precision = int_hits / int_k
            list_precision.append(flt_precision)

            # recall@k
            flt_recall = int_hits / len(set_relevant) if len(set_relevant) > 0 else 0.0
            list_recall.append(flt_recall)

            # ndcg@k
            arr_relevance = np.array([1.0 if item in set_relevant else 0.0 for item in list_recs])
            flt_dcg = np.sum(arr_relevance / np.log2(np.arange(2, len(arr_relevance) + 2)))
            arr_ideal = np.sort(arr_relevance)[::-1]
            flt_idcg = np.sum(arr_ideal / np.log2(np.arange(2, len(arr_ideal) + 2)))
            flt_ndcg = flt_dcg / flt_idcg if flt_idcg > 0 else 0.0
            list_ndcg.append(flt_ndcg)

            # average precision@k
            flt_ap = 0.0
            int_running_hits = 0
            for idx, item in enumerate(list_recs):
                if item in set_relevant:
                    int_running_hits += 1
                    flt_ap += int_running_hits / (idx + 1)
            flt_ap = flt_ap / min(int_k, len(set_relevant)) if len(set_relevant) > 0 else 0.0
            list_ap.append(flt_ap)

            # hit@k
            list_hit.append(1.0 if int_hits > 0 else 0.0)

        # catalog coverage
        int_total_items = len(cls_model.dict_item_to_idx)
        flt_coverage = len(set_recommended_items) / int_total_items if int_total_items > 0 else 0.0

        dict_metrics = {
            f'flt_precision_at_{int_k}': np.mean(list_precision),
            f'flt_recall_at_{int_k}': np.mean(list_recall),
            f'flt_ndcg_at_{int_k}': np.mean(list_ndcg),
            f'flt_map_at_{int_k}': np.mean(list_ap),
            f'flt_hit_rate_at_{int_k}': np.mean(list_hit),
            'flt_coverage': flt_coverage,
        }
        return dict_metrics

## Constants

In [ ]:
str_bucket = 'recommender-system-demo'
str_task = '04_neural_collaborative_filtering'
str_dirname_output = './output'
str_uri = f's3://{str_bucket}/02_split_data/df.parquet'
int_k_at = 10
flt_relevance_threshold = 4.0
str_device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {str_device}')

## Output Directory

In [ ]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

## Import Data

In [ ]:
df = pd.read_parquet(str_uri)
print(f'Dataset: {df.shape[0]:,} rows, {df.shape[1]} columns')
print(f'Data sets: {df["data_set"].value_counts().to_dict()}')

## Train / Validation / Test Split

In [ ]:
df_train = df[df['data_set'] == 'train'].copy()
df_valid = df[df['data_set'] == 'valid'].copy()
df_test = df[df['data_set'] == 'test'].copy()

print(f'Train: {df_train.shape[0]:,} ratings')
print(f'Valid: {df_valid.shape[0]:,} ratings')
print(f'Test:  {df_test.shape[0]:,} ratings')

## Create User and Item Mappings

In [ ]:
# create contiguous ID mappings across all splits
list_all_users = sorted(df['user_id'].unique())
list_all_items = sorted(df['movie_id'].unique())
dict_user_to_idx = {u: i for i, u in enumerate(list_all_users)}
dict_item_to_idx = {m: i for i, m in enumerate(list_all_items)}
int_n_users = len(list_all_users)
int_n_items = len(list_all_items)

# add index columns
df_train['user_idx'] = df_train['user_id'].map(dict_user_to_idx)
df_train['item_idx'] = df_train['movie_id'].map(dict_item_to_idx)
df_valid['user_idx'] = df_valid['user_id'].map(dict_user_to_idx)
df_valid['item_idx'] = df_valid['movie_id'].map(dict_item_to_idx)
df_test['user_idx'] = df_test['user_id'].map(dict_user_to_idx)
df_test['item_idx'] = df_test['movie_id'].map(dict_item_to_idx)

print(f'Users: {int_n_users:,}')
print(f'Items: {int_n_items:,}')

## Bayesian Optimization with Optuna

In [ ]:
dict_best_model = {'cls_model': None, 'flt_rmse': np.inf}

def objective(trial):
    int_embed_dim = trial.suggest_int('int_embed_dim', 8, 128, log=True)
    int_n_layers = trial.suggest_int('int_n_layers', 1, 3)
    list_hidden_dims = []
    int_dim = int_embed_dim * 2
    for i in range(int_n_layers):
        int_dim = max(int_dim // 2, 16)
        list_hidden_dims.append(int_dim)
    flt_dropout = trial.suggest_float('flt_dropout', 0.0, 0.5)
    flt_lr = trial.suggest_float('flt_lr', 1e-4, 1e-2, log=True)
    int_batch_size = trial.suggest_categorical('int_batch_size', [512, 1024, 2048])

    cls_recommender = NCFRecommender(
        int_n_users=int_n_users,
        int_n_items=int_n_items,
        int_embed_dim=int_embed_dim,
        list_hidden_dims=list_hidden_dims,
        flt_dropout=flt_dropout,
        flt_lr=flt_lr,
        int_epochs=15,
        int_batch_size=int_batch_size,
        int_patience=3,
        str_device=str_device,
    )
    cls_recommender.fit(df_train, df_valid, dict_user_to_idx, dict_item_to_idx)

    arr_y_pred = cls_recommender.predict(df_valid['user_id'].values, df_valid['movie_id'].values)
    flt_rmse = np.sqrt(np.mean((df_valid['rating'].values - arr_y_pred) ** 2))

    # track best model
    if flt_rmse < dict_best_model['flt_rmse']:
        dict_best_model['cls_model'] = cls_recommender
        dict_best_model['flt_rmse'] = flt_rmse

    return flt_rmse


int_n_trials = 10
pbar = tqdm(total=int_n_trials, desc='Optuna NCF')

def callback(study, trial):
    pbar.update(1)
    pbar.set_postfix({'best_rmse': f'{study.best_value:.4f}'})

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=int_n_trials, callbacks=[callback])
pbar.close()

print(f'\nBest hyperparameters: {study.best_params}')
print(f'Best validation RMSE: {study.best_value:.4f}')

## Train Final Model

In [ ]:
cls_model_inference = dict_best_model['cls_model']
print(f'Best model: embed_dim={cls_model_inference.int_embed_dim}, '
      f'hidden={cls_model_inference.list_hidden_dims}, '
      f'dropout={cls_model_inference.flt_dropout}')

## Training Loss Curve

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(range(1, len(cls_model_inference.list_train_loss) + 1), cls_model_inference.list_train_loss,
        'o-', label='Train Loss', color='tab:blue')
ax.plot(range(1, len(cls_model_inference.list_valid_loss) + 1), cls_model_inference.list_valid_loss,
        's-', label='Valid Loss', color='tab:orange')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('NCF: Training & Validation Loss', fontsize=14, y=1.02)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{str_dirname_output}/training_loss_curve.png', bbox_inches='tight', dpi=150)
plt.show()

## Rating Prediction Metrics

In [ ]:
cls_evaluator = RecommenderEvaluator(int_k_at=int_k_at, flt_relevance_threshold=flt_relevance_threshold)

# predictions
arr_y_hat_train = cls_model_inference.predict(df_train['user_id'].values, df_train['movie_id'].values)
arr_y_hat_valid = cls_model_inference.predict(df_valid['user_id'].values, df_valid['movie_id'].values)
arr_y_hat_test = cls_model_inference.predict(df_test['user_id'].values, df_test['movie_id'].values)

# rating metrics
dict_train_rating = cls_evaluator.compute_rating_metrics(df_train['rating'].values, arr_y_hat_train)
dict_valid_rating = cls_evaluator.compute_rating_metrics(df_valid['rating'].values, arr_y_hat_valid)
dict_test_rating = cls_evaluator.compute_rating_metrics(df_test['rating'].values, arr_y_hat_test)

print(f'Rating Metrics:')
print(f'  Train - RMSE: {dict_train_rating["flt_rmse"]:.4f}, MAE: {dict_train_rating["flt_mae"]:.4f}')
print(f'  Valid - RMSE: {dict_valid_rating["flt_rmse"]:.4f}, MAE: {dict_valid_rating["flt_mae"]:.4f}')
print(f'  Test  - RMSE: {dict_test_rating["flt_rmse"]:.4f}, MAE: {dict_test_rating["flt_mae"]:.4f}')

## Ranking Metrics

In [ ]:
print('Computing ranking metrics on validation set...')
dict_valid_ranking = cls_evaluator.compute_ranking_metrics(df_valid, cls_model_inference)
for key, val in dict_valid_ranking.items():
    print(f'  Valid {key}: {val:.4f}')

print('\nComputing ranking metrics on test set...')
dict_test_ranking = cls_evaluator.compute_ranking_metrics(df_test, cls_model_inference)
for key, val in dict_test_ranking.items():
    print(f'  Test  {key}: {val:.4f}')

## Save Tuning Results

In [ ]:
dict_tuning = {
    'str_model': 'NCF',
    'int_embed_dim': cls_model_inference.int_embed_dim,
    'str_hidden_dims': str(cls_model_inference.list_hidden_dims),
    'flt_dropout': cls_model_inference.flt_dropout,
    'flt_lr': cls_model_inference.flt_lr,
    'int_batch_size': cls_model_inference.int_batch_size,
    'flt_rmse_train': dict_train_rating['flt_rmse'],
    'flt_rmse_valid': dict_valid_rating['flt_rmse'],
    'flt_rmse_test': dict_test_rating['flt_rmse'],
    'flt_mae_train': dict_train_rating['flt_mae'],
    'flt_mae_valid': dict_valid_rating['flt_mae'],
    'flt_mae_test': dict_test_rating['flt_mae'],
}
dict_tuning.update({f'{k}_valid': v for k, v in dict_valid_ranking.items()})
dict_tuning.update({f'{k}_test': v for k, v in dict_test_ranking.items()})

df_tuning = pd.DataFrame([dict_tuning])
df_tuning.to_csv(f'{str_dirname_output}/df_tuning.csv', index=False)
print('Tuning results saved.')
print(df_tuning.T)

## Save Model

In [ ]:
# save PyTorch state dict
torch.save(cls_model_inference.cls_model.state_dict(), f'{str_dirname_output}/ncf_state_dict.pt')

# save full recommender wrapper (without PyTorch model to avoid pickle issues)
cls_model_inference.cls_model = cls_model_inference.cls_model.cpu()
joblib.dump(cls_model_inference, f'{str_dirname_output}/cls_model_inference.joblib')
print('Model saved.')

## Prediction Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('NCF: Prediction Analysis', fontsize=14, y=1.02)

# prediction distribution
axes[0].hist(arr_y_hat_test, bins=50, color='tab:blue', edgecolor='black', alpha=0.7, label='Predicted')
axes[0].hist(df_test['rating'].values, bins=5, color='tab:red', edgecolor='black', alpha=0.4, label='Actual')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')
axes[0].set_title('Predicted vs Actual Rating Distribution (Test)')
axes[0].legend()

# residual distribution
arr_residuals = df_test['rating'].values - arr_y_hat_test
axes[1].hist(arr_residuals, bins=50, color='tab:orange', edgecolor='black', alpha=0.7)
axes[1].axvline(x=0, color='black', linestyle='--', linewidth=1)
axes[1].set_xlabel('Residual (Actual - Predicted)')
axes[1].set_ylabel('Count')
axes[1].set_title(f'Residual Distribution (Test) | Mean: {arr_residuals.mean():.3f}')

plt.tight_layout()
plt.savefig(f'{str_dirname_output}/prediction_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

## Save Test Predictions

In [ ]:
df_test_preds = pd.DataFrame({
    'user_id': df_test['user_id'].values,
    'movie_id': df_test['movie_id'].values,
    'rating': df_test['rating'].values,
    'y_hat': arr_y_hat_test,
})
df_test_preds.to_parquet(f'{str_dirname_output}/test_predictions.parquet', index=False)
print(f'Test predictions saved: {df_test_preds.shape[0]:,} rows')